# Notebook 0 — Verificación del Entorno Spark + HDFS
**Procesamiento de Alto Volúmen de Datos — Pontificia Universidad Javeriana**

> Ejecutar este notebook PRIMERO para confirmar que el clúster está operativo antes de los experimentos.

## 0.1 Configurar variables de entorno

In [11]:
import os

# ─── Ajusta estas rutas a tu instalación ───────────────────────────────────
MASTER_IP       = '10.43.97.145'          # IP del nodo Master
SPARK_MASTER    = f'spark://{MASTER_IP}:7077'
HDFS_URI        = f'hdfs://{MASTER_IP}:9000'
CONDA_ENV       = '/opt/conda_envs/pyspark_env'
HADOOP_HOME     = '/nfs/condor/apps/hadoop-3.5.0'
JAVA_HOME       = '/usr/lib/jvm/java-17-openjdk'
# ───────────────────────────────────────────────────────────────────────────

# os.environ['JAVA_HOME']            = JAVA_HOME
# os.environ['HADOOP_HOME']          = HADOOP_HOME
# os.environ['HADOOP_CONF_DIR']      = f'{HADOOP_HOME}/etc/hadoop'
# os.environ['PYSPARK_PYTHON']       = f'{CONDA_ENV}/bin/python'
# os.environ['PYSPARK_DRIVER_PYTHON']= f'{CONDA_ENV}/bin/python'

print('Variables de entorno configuradas')
print(f'   SPARK_MASTER : {SPARK_MASTER}')
print(f'   HDFS_URI     : {HDFS_URI}')

Variables de entorno configuradas
   SPARK_MASTER : spark://10.43.97.145:7077
   HDFS_URI     : hdfs://10.43.97.145:9000


## 0.2 Verificar versiones

In [12]:
import pyspark
import pandas as pd
import platform

print(f'Python   : {platform.python_version()}')
print(f'PySpark  : {pyspark.__version__}')
print(f'Pandas   : {pd.__version__}')

Python   : 3.11.15
PySpark  : 3.5.2
Pandas   : 3.0.3


## 0.3 Crear SparkSession conectada al clúster y a HDFS

In [27]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('Setup_Verificacion')
    .master(SPARK_MASTER)
    # Todos los workers usan el mismo entorno conda
    .config('spark.pyspark.python', f'{CONDA_ENV}/bin/python')
    # .config('spark.executor.memory', '2g')
    # .config('spark.driver.memory', '2g')
    # Integración con HDFS
    .config('spark.hadoop.fs.defaultFS', HDFS_URI)
    .getOrCreate()
)

sc = spark.sparkContext
print(f' SparkSession creada')
print(f'   Spark version  : {spark.version}')
print(f'   Master         : {sc.master}')
print(f'   App Name       : {sc.appName}')
print(f'\n Spark UI disponible en: http://{MASTER_IP}:4040')

 SparkSession creada
   Spark version  : 3.5.2
   Master         : spark://10.43.97.145:7077
   App Name       : Setup_Verificacion

 Spark UI disponible en: http://10.43.97.145:4040


## 0.4 Verificar acceso a HDFS

In [15]:
# Crear estructura de directorios en HDFS para los experimentos
import subprocess

hdfs = f'{HADOOP_HOME}/bin/hdfs'

dirs = [
    '/experimentos',
    '/experimentos/datos_entrada',
    '/experimentos/resultados',
    '/experimentos/exp1_io',
    '/experimentos/exp2_wordcount',
    '/experimentos/exp3_sql',
    '/experimentos/exp4_fallos',
]

for d in dirs:
    result = subprocess.run([hdfs, 'dfs', '-mkdir', '-p', d],
                            capture_output=True, text=True)
    status = 'Ok' if result.returncode == 0 else '(ya existe)'
    print(f'{status} {d}')

# Listar raíz de HDFS
print('\nContenido de / en HDFS:')
result = subprocess.run([hdfs, 'dfs', '-ls', '/'], capture_output=True, text=True)
print(result.stdout)

Ok /experimentos
Ok /experimentos/datos_entrada
Ok /experimentos/resultados
Ok /experimentos/exp1_io
Ok /experimentos/exp2_wordcount
Ok /experimentos/exp3_sql
Ok /experimentos/exp4_fallos

Contenido de / en HDFS:
Found 1 items
drwxr-xr-x   - estudiante supergroup          0 2026-05-21 18:43 /experimentos



## 0.5 Test rápido: Spark lee desde HDFS

In [34]:
# Cambia temporalmente el master para confirmar que Spark funciona
# Escribir un CSV pequeño a HDFS
test_csv = '/tmp/test_setup.csv'
with open(test_csv, 'w') as f:
    f.write('id,valor\n')
    for i in range(100):
        f.write(f'{i},{i*2}\n')

spark_local = (
    SparkSession.builder
    .appName('test_local')
    .master('local[*]')   # ← sin clúster, todo en el driver
    .getOrCreate()
)
df = spark_local.read.csv('/tmp/test_setup.csv', header=True, inferSchema=True)
print(df.count())   # Si esto funciona, el problema es el clúster, no HDFS/Python
spark_local.stop()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: hdfs://10.43.97.145:9000/tmp/test_setup.csv.

In [31]:
import subprocess, os

# Escribir un CSV pequeño a HDFS
test_csv = '/tmp/test_setup.csv'
with open(test_csv, 'w') as f:
    f.write('id,valor\n')
    for i in range(100):
        f.write(f'{i},{i*2}\n')

subprocess.run([f'{HADOOP_HOME}/bin/hdfs', 'dfs', '-put', '-f',
                test_csv, '/experimentos/datos_entrada/'], check=True)

# Leer desde HDFS con Spark
df = spark.read.csv(f'{HDFS_URI}/experimentos/datos_entrada/test_setup.csv',
                    header=True, inferSchema=True)
print(f'Filas leídas desde HDFS: {df.count()}')
df.show(5)
print('\n Entorno Spark + HDFS operativo. Puedes continuar con los experimentos.')

2026-05-21 19:24:36,167 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:24:36,194 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:24:36,696 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:24:36,717 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:24:36,920 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:24:36,940 INFO Configuration.deprecation: dfs.permissions in hdfs-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19:24:36,999 INFO Configuration.deprecation: dfs.permissions in core-site.xml is deprecated. Instead, use dfs.permissions.enabled
2026-05-21 19

KeyboardInterrupt: 

In [32]:
spark.stop()
print('SparkSession detenida.')

SparkSession detenida.
